# 08e -- Manual Extraction Rankings

**Purpose**: Ingest manually-extracted ranking sheets from
`RookieRankings_2026_ManualExtraction.xlsx` into `fact_rookie_rankings`.
Each Excel sheet is one `source_site` / `source_name`.

| Sheet | source_name | source_site | phase |
|---|---|---|---|
| rotoballer | RotoBaller | RotoBaller | post_draft |
| mystery_iono | mystery_iono | mystery_iono | post_draft |
| dynastyleaguefootball | DynastyLeagueFootball | DynastyLeagueFootball | pre_combine |
| fantasycalc | FantasyCalc | FantasyCalc | post_draft |
| fantasypros_idp | FantasyPros_IDP | FantasyPros | post_draft |

**DLF phase note**: `Rank_Date` is 2025-12-31, which is pre-combine — using `pre_combine`.

**FP IDP note**: FantasyPros IDP rookie rankings cannot be scraped via `requests` —
the page at `dynasty-rookies-idp.php` serves a full veteran draft board in its raw HTML
`ecrData`; the actual defensive rookie table is rendered client-side only. Data is
manually extracted into the Excel sheet instead. Migrated from 08a.

**Prerequisites**: `08_fact_rookie_rankings_seed.ipynb`, `01_dim_rookie_prospect.ipynb`

**Outputs**: `data/fact_rookie_rankings.parquet` (appended),
             `data/review_fuzzy_matches.csv` (if new review items found)

## 1 -- Setup & Config

In [ ]:
# ---- Setup: shared helpers + config from etl_helpers ------------------------
import re
import shutil
import tempfile
from pathlib import Path

import pandas as pd

import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (
    LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS,
    clean_player_name, generate_player_key,
    add_players_from_source, ingest_ranking_source, append_review,
)

# Excel is in the same data dir; copy to a portable temp file in case it is open in Excel.
XLSX_PATH = DATA / "RookieRankings_2026_ManualExtraction.xlsx"
XLSX_TMP  = Path(tempfile.gettempdir()) / "RookieRankings_manual_tmp.xlsx"

## 2 -- Shared Helpers

In [ ]:
# clean_player_name / generate_player_key are imported from etl_helpers.
assert clean_player_name("J.K. Dobbins") == "jk dobbins"
assert clean_player_name("D'Andre Swift") == "d'andre swift"
print("Helpers OK (etl_helpers)")

## 3 -- add_players_from_source

In [ ]:
# add_players_from_source is imported from etl_helpers (canonical, alias-aware).

## 4 -- ingest_ranking_source

In [ ]:
# ingest_ranking_source is imported from etl_helpers.

## 5 -- Sheet Parsers

In [ ]:
def _pos_rank_from_group(df: pd.DataFrame, pos_col: str = "position_raw", rank_col: str = "global_rank") -> pd.Series:
    """Rank players within each position group by their global_rank order."""
    return df.groupby(pos_col)[rank_col].rank(method="first").astype(int)


def parse_rotoballer(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """Tier/Rank list; positional_rank computed from order within Pos group."""
    out = pd.DataFrame()
    out["player_name"]     = df["Player Name"].str.strip()
    out["position_raw"]    = df["Pos"].str.strip().str.upper()
    out["global_rank"]     = df["Rank"].astype(int)
    out["grade"]           = df["Rank"].astype(float)
    out["positional_rank"] = _pos_rank_from_group(out)
    rank_date = pd.to_datetime(df["Date"].iloc[0]).strftime("%Y-%m-%d")
    return out, rank_date


def _invert_lastname_firstname(s: str) -> str:
    # "Love, Jeremiyah" → "Jeremiyah Love"; pass-through if no comma.
    if pd.isna(s):
        return ""
    parts = [p.strip() for p in str(s).split(",", 1)]
    return f"{parts[1]} {parts[0]}" if len(parts) == 2 else parts[0]


def parse_mystery_iono(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """Name field is 'Lastname, Firstname'; invert before matching."""
    out = pd.DataFrame()
    out["player_name"]     = df["lastname_firstname"].apply(_invert_lastname_firstname)
    out["position_raw"]    = df["position_raw"].str.strip().str.upper()
    out["global_rank"]     = df["rank"].astype(int)
    out["grade"]           = df["rank"].astype(float)
    out["positional_rank"] = _pos_rank_from_group(out)
    rank_date = pd.to_datetime(df["date_added"].iloc[0]).strftime("%Y-%m-%d")
    return out, rank_date


def parse_dynastyleaguefootball(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """Pos column encodes both position and positional_rank: 'RB1', 'WR2', etc."""
    out = pd.DataFrame()
    out["player_name"]  = df["Name"].str.strip()
    pos_series          = df["Pos"].str.strip()
    # Extract alphabetic prefix → canonical position; extract trailing digit → pos rank.
    out["position_raw"]    = pos_series.str.extract(r"^([A-Za-z]+)")[0].str.upper()
    out["positional_rank"] = pos_series.str.extract(r"(\d+)$")[0].fillna(1).astype(int)
    out["global_rank"]     = df["Rank"].astype(int)
    out["grade"]           = df["Avg"].astype(float)   # consensus avg rank across 6 experts
    rank_date = pd.to_datetime(df["Rank_Date"].iloc[0]).strftime("%Y-%m-%d")
    return out, rank_date


def parse_fantasycalc(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """overallRank / positionRank are direct; grade = overallRank (no trade value stored here)."""
    out = pd.DataFrame()
    out["player_name"]     = df["name"].str.strip()
    out["position_raw"]    = df["position"].str.strip().str.upper()
    out["global_rank"]     = df["overallRank"].astype(int)
    out["positional_rank"] = df["positionRank"].astype(int)
    out["grade"]           = df["overallRank"].astype(float)
    rank_date = pd.to_datetime(df["date_added"].iloc[0]).strftime("%Y-%m-%d")
    return out, rank_date


def parse_fantasypros_idp(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """Manually extracted from FantasyPros dynasty-rookies-idp.php (client-side only).
    Player Name has '\\xa0(TEAM)' suffix; POS encodes position + positional_rank ('LB1', 'EDGE2')."""
    out = pd.DataFrame()
    # Strip non-breaking space + team abbreviation: "Sonny Styles\xa0(WAS)" → "Sonny Styles"
    out["player_name"]  = df["Player Name"].str.replace(r"\s*\(.*?\)\s*$", "", regex=True).str.strip()
    pos_series          = df["POS"].str.strip()
    out["position_raw"]    = pos_series.str.extract(r"^([A-Za-z]+)")[0].str.upper()
    out["positional_rank"] = pos_series.str.extract(r"(\d+)$")[0].fillna(1).astype(int)
    out["global_rank"]     = df["RK"].astype(int)
    out["grade"]           = df["RK"].astype(float)
    # No date column in this sheet — use today as rank_date.
    rank_date = TODAY
    return out, rank_date

## 6 -- Sources Config

In [ ]:
SOURCES = {
    "RotoBaller": {
        "sheet":       "rotoballer",
        "source_site": "RotoBaller",
        "phase":       "post_draft",
        "parser":      parse_rotoballer,
    },
    "mystery_iono": {
        "sheet":       "mystery_iono",
        "source_site": "mystery_iono",
        "phase":       "post_draft",
        "parser":      parse_mystery_iono,
    },
    "DynastyLeagueFootball": {
        "sheet":       "dynastyleaguefootball",
        "source_site": "DynastyLeagueFootball",
        "phase":       "pre_combine",  # Rank_Date=2025-12-31 → pre-combine
        "parser":      parse_dynastyleaguefootball,
    },
    "FantasyCalc": {
        "sheet":       "fantasycalc",
        "source_site": "FantasyCalc",
        "phase":       "post_draft",
        "parser":      parse_fantasycalc,
    },
    "FantasyPros_IDP": {
        "sheet":       "fantasypros_idp",
        "source_site": "FantasyPros",
        "phase":       "post_draft",
        "parser":      parse_fantasypros_idp,
    },
}

## 7 -- Ingestion Loop

In [ ]:
shutil.copy2(XLSX_PATH, XLSX_TMP)
xl = pd.ExcelFile(XLSX_TMP)

_all_reviews: list[pd.DataFrame] = []
_failed: list[str] = []

for source_name, cfg in SOURCES.items():
    print("=" * 60)
    print(f"Processing: {source_name}  [{cfg['phase']}]")

    try:
        df_raw = xl.parse(cfg["sheet"])
        df, rank_date = cfg["parser"](df_raw)
        print(f"  Parsed {len(df)} rows  |  rank_date={rank_date}")
    except Exception as e:
        print(f"  ERROR parsing: {e}")
        _failed.append(source_name)
        continue

    try:
        _, review = add_players_from_source(df, source_name=source_name)
        if not review.empty:
            _all_reviews.append(review)
    except Exception as e:
        print(f"  ERROR add_players: {e}")
        _failed.append(source_name)
        continue

    try:
        ingest_ranking_source(
            df,
            source_name=source_name,
            source_site=cfg["source_site"],
            phase=cfg["phase"],
            rank_date=rank_date,
        )
    except Exception as e:
        print(f"  ERROR ingesting: {e}")
        _failed.append(source_name)

if _failed:
    print(f"\nFAILED sources: {_failed}")
else:
    print("\nAll sources ingested successfully.")

## 8 -- Write Review CSV

In [ ]:
append_review(_all_reviews)

## 9 -- Validation

In [ ]:
fr  = pd.read_parquet(DATA / "fact_rookie_rankings.parquet")
sub = fr[fr["source_name"].isin(SOURCES.keys())]
print(f"Rows for this run : {len(sub)}")
print(sub.groupby(["source_name", "phase"]).size().rename("rows").to_string())
print()
print(f"fact_rookie_rankings total : {len(fr)} rows")
dupes = fr.duplicated(subset=["player_key", "source_name", "phase", "draft_year"]).sum()
print(f"Composite key duplicates   : {dupes} (expected 0)")
print()
print("Source breakdown:")
print(fr.groupby(["source_site", "source_name", "phase"]).size().rename("rows").to_string())